# Task027 semantic rule discovery

This notebook replaces the previous visible-patch workflow with a train-only semantic rule workflow.

Goal:
- infer from `train` examples only;
- validate on `test + arc-gen`;
- avoid using `test/arc-gen` to calibrate a patch bank;
- document why the old 11×11 patch-detector workflow got zero hidden score.

In [1]:
import json
from pathlib import Path
from collections import Counter
import numpy as np

TASK_ID = 'task027'
MODEL_VERSION = 'task027-semantic-rot180-square-bbox-rule'
TASK_JSON = next((p for p in [
    Path('Co_Kaggle/g3/competition_material/taskfiles/task027.json'),
    Path('competition_material/taskfiles/task027.json'),
    Path('/mnt/data/task027.json'),
] if p.exists()), None)
assert TASK_JSON is not None, 'Missing task027.json'

task = json.load(open(TASK_JSON, encoding='utf-8'))
print('MODEL_VERSION:', MODEL_VERSION)
print('TASK_JSON:', TASK_JSON)
print('examples:', {k: len(task.get(k, [])) for k in ['train', 'test', 'arc-gen']})
print('shape modes:', Counter(f"{len(ex['input'])}x{len(ex['input'][0])}" for split in ['train','test','arc-gen'] for ex in task.get(split, [])).most_common())

MODEL_VERSION: task027-semantic-rot180-square-bbox-rule
TASK_JSON: /mnt/data/task027.json
examples: {'train': 3, 'test': 1, 'arc-gen': 261}
shape modes: [('10x10', 265)]


In [2]:
def coords(grid, value=1):
    return {(r, c) for r, row in enumerate(grid) for c, x in enumerate(row) if x == value}

def bbox(points):
    rs = [r for r, c in points]
    cs = [c for r, c in points]
    return min(rs), min(cs), max(rs), max(cs)

def infer_square_bbox_from_ones(grid):
    # Train-derived semantic rule:
    # 1. find the bbox of color-1 cells;
    # 2. extend it to a square;
    # 3. if height > width, extend left with right boundary fixed;
    # 4. if width > height, extend downward with top boundary fixed.
    pts = coords(grid, 1)
    if not pts:
        return None
    r0, c0, r1, c1 = bbox(pts)
    h = r1 - r0 + 1
    w = c1 - c0 + 1
    if h > w:
        side = h
        return r0, c1 - side + 1, r1, c1
    if w > h:
        side = w
        return r0, c0, r0 + side - 1, c1
    return r0, c0, r1, c1

def predict_task027_semantic(grid):
    arr = np.array(grid, dtype=int)
    square = infer_square_bbox_from_ones(grid)
    if square is None:
        return arr.tolist()

    r0, c0, r1, c1 = square
    out = arr.copy()
    for r, c in coords(grid, 1):
        rr = r0 + r1 - r
        cc = c0 + c1 - c
        if 0 <= rr < out.shape[0] and 0 <= cc < out.shape[1] and out[rr, cc] == 0:
            out[rr, cc] = 2
    return out.tolist()

def eval_grid(fn, splits=('train','test','arc-gen')):
    rows = []
    right = total = 0
    first_wrong = None
    for split in splits:
        sr = st = 0
        for i, ex in enumerate(task.get(split, [])):
            pred = fn(ex['input'])
            ok = pred == ex['output']
            sr += int(ok)
            st += 1
            right += int(ok)
            total += 1
            if not ok and first_wrong is None:
                first_wrong = {'split': split, 'index': i}
        rows.append((split, sr, st, sr / st if st else None))
    return {'right': right, 'total': total, 'accuracy': right / total if total else None, 'first_wrong': first_wrong, 'rows': rows}

semantic_summary = eval_grid(predict_task027_semantic)
print('semantic rule accuracy:', semantic_summary['right'], '/', semantic_summary['total'], semantic_summary['accuracy'], 'first_wrong=', semantic_summary['first_wrong'])
print('rows:', semantic_summary['rows'])
assert semantic_summary['right'] == semantic_summary['total']

semantic rule accuracy: 265 / 265 1.0 first_wrong= None
rows: [('train', 3, 3, 1.0), ('test', 1, 1, 1.0), ('arc-gen', 261, 261, 1.0)]


In [3]:
# Diagnostic: what happens if the old 11x11 patch-detector is trained on train only?
# This explains why visible-perfect patch banks can fail on Kaggle hidden examples.

RAD = 5

def patch_feat_01(grid, r, c, rad=RAD):
    a = np.asarray(grid)
    h, w = a.shape
    f = np.zeros((2, 2 * rad + 1, 2 * rad + 1), np.int8)
    for i, dr in enumerate(range(-rad, rad + 1)):
        for j, dc in enumerate(range(-rad, rad + 1)):
            rr, cc = r + dr, c + dc
            if 0 <= rr < h and 0 <= cc < w:
                col = int(a[rr, cc])
                if col in (0, 1):
                    f[col, i, j] = 1
    return tuple(f.flatten().tolist())

mapping = {}
conflicts = []
for ei, ex in enumerate(task['train']):
    target = (np.asarray(ex['output']) == 2).astype(np.int8)
    for r in range(10):
        for c in range(10):
            key = patch_feat_01(ex['input'], r, c)
            y = int(target[r, c])
            if key in mapping and mapping[key] != y:
                conflicts.append((ei, r, c, mapping[key], y))
            else:
                mapping[key] = y

def pred_patch_train_only(grid):
    out = np.asarray(grid).copy()
    for r in range(10):
        for c in range(10):
            if mapping.get(patch_feat_01(grid, r, c), 0):
                out[r, c] = 2
    return out.tolist()

patch_summary = eval_grid(pred_patch_train_only)
print('11x11 train-only patch-detector accuracy:', patch_summary['right'], '/', patch_summary['total'], patch_summary['accuracy'], 'first_wrong=', patch_summary['first_wrong'])
print('rows:', patch_summary['rows'])
print('unique train patch patterns:', len(mapping))
print('positive train patch-detectors:', sum(mapping.values()))
print('conflicts:', len(conflicts))

11x11 train-only patch-detector accuracy: 3 / 265 0.011320754716981131 first_wrong= {'split': 'test', 'index': 0}
rows: [('train', 3, 3, 1.0), ('test', 0, 1, 0.0), ('arc-gen', 0, 261, 0.0)]
unique train patch patterns: 300
positive train patch-detectors: 28
conflicts: 0


In [4]:
# Submission status.
# The semantic rule is compact and generalizes across all visible generated examples.
# A proper Kaggle submission still needs an ONNX compiler for this rule.
# Do NOT use the old visible-trained 11x11 detector as the hidden submission model.

final_row = {
    'task_id': TASK_ID,
    'model_version': MODEL_VERSION,
    'semantic_visible_right': semantic_summary['right'],
    'semantic_visible_total': semantic_summary['total'],
    'semantic_visible_accuracy': semantic_summary['accuracy'],
    'patch_train_only_right': patch_summary['right'],
    'patch_train_only_total': patch_summary['total'],
    'patch_train_only_accuracy': patch_summary['accuracy'],
    'status': 'semantic_rule_validated_no_onnx_export_yet',
}
print(final_row)

{'task_id': 'task027', 'model_version': 'task027-semantic-rot180-square-bbox-rule', 'semantic_visible_right': 265, 'semantic_visible_total': 265, 'semantic_visible_accuracy': 1.0, 'patch_train_only_right': 3, 'patch_train_only_total': 265, 'patch_train_only_accuracy': 0.011320754716981131, 'status': 'semantic_rule_validated_no_onnx_export_yet'}
